In [1]:
from keras.models import Sequential
from keras.layers import Activation,LSTM,Dense
from tensorflow.keras.optimizers import Adam

In [2]:
import pandas as pd
import numpy as np
import re

In [3]:
rap_df = pd.read_csv("./../dataset/preprocessed_rap.csv")

In [4]:
rap_df.shape[0]

10077

In [5]:
rap_df = rap_df.drop('Unnamed: 0', axis=1)
rap_df = rap_df.drop('Unnamed: 0.1', axis=1)

In [6]:
rap_df.head(5)
# rap_df.loc[0]["lyrics"]

,artist,genre,title,lyrics,word_num
0,Snoop Dogg,rap,Gin and Juice,"(Ugh) Ha-ha-ha, I'm serious, nigga One of y'al...",618
1,Snoop Dogg,rap,Drop It Like It’s Hot,"Snoop Snoop When the pimp's in the crib, ma D...",781
2,Snoop Dogg,rap,Ain’t No Fun (If the Homies Can’t Have None),You're back now at the jack-off hour This is D...,599
3,Snoop Dogg,rap,Murder Was the Case (Death After Visualizing E...,"(""Indo Smoke"" Plays in Background) Aye, aye, ...",631
4,Snoop Dogg,rap,Who Am I (What’s My Name)?,EeeyiyiyiyiyahtheDoggPound'sinthehou-owwse (th...,428


In [7]:
train_sz = 500
train_rap = rap_df.iloc[:train_sz]
print(train_rap.shape[0])

500


In [30]:
def split_str(delimiters, string, maxsplit=0):
    regex_pattern = '|'.join(map(re.escape, delimiters))
    return re.split(regex_pattern, string, maxsplit)

In [188]:
corpus=''
for index,row in train_rap.iterrows():
    corpus += " "
    # corpus += row["title"]
    lyric = split_str(" ;,-'\\\'\"()[].?!:{}",  row["lyrics"].lower().strip())
    lyric = [s for s in lyric if s!=''][0:100]
    corpus += " " + " ".join(lyric)

In [29]:
# vocab=list(set(corpus.lower().strip().split(" ")))
# vocab = re.split('; |, |\*|\n',corpus)
cleaned_corpus = split_str(" ;,-'\\\'\"()[].?!:{}", corpus.lower().strip())
cleaned_corpus.extend([";", ",", "-", "\\", "'", '"', "(", ")", "[", "]",  ".", "?", "!", ":", "{", "}"])
# print(cleaned_corpus)
vocab = list(set(cleaned_corpus))
print(len(vocab))
# print(vocab)
char_ix={c:i for i,c in enumerate(vocab)}
ix_char={i:c for i,c in enumerate(vocab)}

NameError: name 'split_str' is not defined

### Corpus-2

In [21]:
corpus = ''
for index, row in train_rap.iterrows():
    
    corpus += row["lyrics"].lower()
vocab = list(set([x for x in corpus]))

In [12]:
list(set(train_rap.loc[0]["lyrics"].lower()))

['d',
 'a',
 'g',
 ' ',
 'p',
 '(',
 'b',
 '.',
 'q',
 'o',
 'l',
 '0',
 'k',
 '‘',
 'h',
 'n',
 'f',
 'u',
 's',
 ';',
 'x',
 'c',
 'w',
 'r',
 ')',
 'j',
 'i',
 't',
 '?',
 'z',
 ',',
 '8',
 '-',
 'y',
 'e',
 '…',
 'm',
 "'",
 'v']

In [23]:
vocab.sort()
# print(vocab)
print("e" in vocab)
char_ix={c:i for i,c in enumerate(vocab)}
ix_char={i:c for i,c in enumerate(vocab)}

True


In [21]:
import json
with open("./../dataset/char_ix.json", "w") as write_file:
    json.dump(char_ix, write_file, indent=4)
with open("./../dataset/ix_char.json", "w") as write_file:
    json.dump(ix_char, write_file, indent=4)

In [24]:
char_ix["u"]
# 7000 - 109776
# 6000 - 97913
# 3000 - 67438

51

In [130]:
ix_char

{0: '',
 1: 'search',
 2: 'rick',
 3: 'kick',
 4: 'heard',
 5: 'mission',
 6: 'tonners',
 7: 'riding\u2005',
 8: 'wisdom',
 9: 'years',
 10: 's\u2005a',
 11: 'must',
 12: 'second',
 13: 'pen',
 14: 'kaki',
 15: 'thrillin',
 16: 'old‚',
 17: 'chronic',
 18: 'shootin',
 19: 'backin',
 20: 'dressed',
 21: 'dtf',
 22: 'haters',
 23: '\u2005there',
 24: 'of\u2005people',
 25: ';',
 26: 'hill',
 27: 'freeze',
 28: 'grabs',
 29: 'able',
 30: 'hoe',
 31: 'cologne',
 32: 'dub',
 33: 'mine',
 34: 'along',
 35: 'let',
 36: 'far',
 37: 'upside',
 38: 'slapper',
 39: 'nah',
 40: 'trippin',
 41: 'ignat',
 42: 'nursed',
 43: 'given',
 44: 'de',
 45: 'al',
 46: 'impala',
 47: 'china',
 48: 'place',
 49: 'shortie',
 50: 'perfect',
 51: 'fact',
 52: 'caution',
 53: 'toe',
 54: 'ages',
 55: 'adams',
 56: 'christmas',
 57: 'ain',
 58: 'greener',
 59: 'rich',
 60: 'caked',
 61: 'clockin',
 62: 'always',
 63: 'creeping',
 64: 'radio',
 65: 'spea',
 66: 'rappin',
 67: 'net',
 68: 'creepin',
 69: 'pimp',
 70:

In [27]:
maxlen=40
vocab_size=len(vocab) # 72

In [41]:
sentences=[]
next_word=[]
num_words = 40
for index, row in train_rap.iterrows():
    # lyric = split_str(" ;,-'\\\'\"()[].?!:{}",  row["lyrics"].lower().strip())
    # lyric = [s for s in lyric if s!=''][0:100]

    # lyric = row["lyrics"].lower().split(" ")[0:100]
    lyric = row["lyrics"].lower()[0:100]
    # print(lyric[0:40])
    for i in range(len(lyric)-num_words-1):        
        sentences.append("".join(lyric[i:i+num_words]))
        next_word.append(lyric[i+num_words])

In [42]:
print(len(sentences))
sentences

29500


["(ugh) ha-ha-ha, i'm serious, nigga one o",
 "ugh) ha-ha-ha, i'm serious, nigga one of",
 "gh) ha-ha-ha, i'm serious, nigga one of ",
 "h) ha-ha-ha, i'm serious, nigga one of y",
 ") ha-ha-ha, i'm serious, nigga one of y'",
 " ha-ha-ha, i'm serious, nigga one of y'a",
 "ha-ha-ha, i'm serious, nigga one of y'al",
 "a-ha-ha, i'm serious, nigga one of y'all",
 "-ha-ha, i'm serious, nigga one of y'all ",
 "ha-ha, i'm serious, nigga one of y'all n",
 "a-ha, i'm serious, nigga one of y'all ni",
 "-ha, i'm serious, nigga one of y'all nig",
 "ha, i'm serious, nigga one of y'all nigg",
 "a, i'm serious, nigga one of y'all nigga",
 ", i'm serious, nigga one of y'all niggas",
 " i'm serious, nigga one of y'all niggas ",
 "i'm serious, nigga one of y'all niggas g",
 "'m serious, nigga one of y'all niggas go",
 "m serious, nigga one of y'all niggas got",
 " serious, nigga one of y'all niggas got ",
 "serious, nigga one of y'all niggas got s",
 "erious, nigga one of y'all niggas got so",
 "rious, n

In [43]:
next_word

['f',
 ' ',
 'y',
 "'",
 'a',
 'l',
 'l',
 ' ',
 'n',
 'i',
 'g',
 'g',
 'a',
 's',
 ' ',
 'g',
 'o',
 't',
 ' ',
 's',
 'o',
 'm',
 'e',
 ' ',
 'b',
 'a',
 'd',
 ' ',
 'm',
 'o',
 't',
 'h',
 'e',
 'r',
 'f',
 'u',
 'c',
 'k',
 'i',
 'n',
 "'",
 ' ',
 'b',
 'r',
 'e',
 'a',
 't',
 'h',
 ' ',
 '(',
 'o',
 'h',
 ',',
 ' ',
 'm',
 'a',
 'n',
 ')',
 ' ',
 ',',
 ' ',
 'm',
 'a',
 ' ',
 'd',
 'r',
 'o',
 'p',
 ' ',
 'i',
 't',
 ' ',
 'l',
 'i',
 'k',
 'e',
 ' ',
 'i',
 't',
 "'",
 's',
 ' ',
 'h',
 'o',
 't',
 ',',
 ' ',
 'd',
 'r',
 'o',
 'p',
 ' ',
 'i',
 't',
 ' ',
 'l',
 'i',
 'k',
 'e',
 ' ',
 'i',
 't',
 "'",
 's',
 ' ',
 'h',
 'o',
 't',
 ',',
 ' ',
 'd',
 'r',
 'o',
 'p',
 ' ',
 'i',
 't',
 ' ',
 's',
 ' ',
 'i',
 's',
 ' ',
 'd',
 'j',
 ' ',
 'e',
 'z',
 ' ',
 'd',
 'i',
 'c',
 'c',
 ' ',
 'o',
 'n',
 ' ',
 'w',
 '-',
 'b',
 'a',
 'l',
 'l',
 's',
 ' ',
 'r',
 'i',
 'g',
 'h',
 't',
 ' ',
 'n',
 'o',
 'w',
 ' ',
 's',
 'o',
 'm',
 'e',
 't',
 'h',
 'i',
 'n',
 'g',
 ' ',
 'n',
 'e'

### Glove Embeddings

In [17]:
f = open("./../dataset/glove.6B.100d.txt", encoding="utf8") 
embedd_index = {}
for line in f:
  val = line.split()
  word = val[0]
  coff = np.asarray(val[1:],dtype = 'float')
  embedd_index[word] = coff
f.close()

In [74]:
def get_word_embed(sent, num_words, vec_size):
#     if flag:
#         print("func first")
    # words = [d[0] for d in dp]
    words = sent.split(" ")
    sent_len = len(words)

    pad_total = num_words-sent_len
    # print(pad_total)
    pad_right = int(pad_total/2)
    pad_left = pad_total - pad_right

    word_vector = np.zeros((num_words,vec_size), dtype=float)

    for i in range(pad_left, pad_left + sent_len):
        word = words[i-pad_left].lower()
        try:
            word_vector[i,:] = embedd_index[word][:]
        except:
            word_vector[i, :] = np.ones(vec_size)

    return word_vector

In [71]:
len(sentences)

17720

In [182]:
vec = get_word_embed(sentences[0], num_words, 100)
print(sentences[0].split(" "))

['ugh', 'ha', 'ha', 'ha', 'i', 'm', 'serious', 'nigga', 'one', 'of', 'y', 'all', 'niggas', 'got', 'some']


In [117]:
char_ix['ugh']

2045

In [184]:
print(vocab_size)

8002


In [44]:
X=np.zeros((len(sentences),num_words,vocab_size))
y=np.zeros((len(sentences),vocab_size))
for ix in range(len(sentences)):
    if ix%5000==0:
        print(ix)
        print(sentences[ix])
    y[ix,char_ix[next_word[ix]]]=1
    
    words = split_str(" ;,-'\\\'\"()[].?!:{}", sentences[ix].strip())
    # words = sentences[ix].split(" ")
    # print(ix, sentences[ix])
    for iy in range(maxlen):
        X[ix,iy,char_ix[sentences[ix][iy]]]=1
        # X[ix,iy,char_ix[words[iy]]]=1

0
(ugh) ha-ha-ha, i'm serious, nigga one o
5000
n that plays only platinum hits, w ballz
10000
op (rick rock beats) are you ready for s
15000
o) (sephgotthewaves)  yeah, thinkin' 'bo
20000
 up the line guys hit you up from a blin
25000
 lord knows (coughing) lord knows lord k


In [45]:
print(X.shape, y.shape)

(29500, 40, 172) (29500, 172)


In [75]:
vec_size = 100
X=np.zeros((len(sentences),num_words,vec_size))
y=np.zeros((len(sentences),vec_size))
for ix in range(len(sentences)):
    if ix%5000==0:
        print(ix)
        print(sentences[ix])
    try:
        y[ix] = embedd_index[next_word[ix]]
        X[ix] = get_word_embed(sentences[ix], num_words, vec_size)
    except:
        y[ix] = np.ones(vec_size, dtype=np.float)
        X[ix] = get_word_embed(sentences[ix], num_words, vec_size)
    # y[ix,char_ix[next_char[ix]]]=1
    # for iy in range(maxlen):
    #     X[ix,iy,char_ix[sentences[ix][iy]]]=1

0
(Ugh) Ha-ha-ha, I'm serious, nigga One of y'all niggas got
5000
my life might be like Growin up in the street


C:\Users\ramyasri palti\AppData\Local\Temp\ipykernel_28700\1127628660.py:12: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  y[ix] = np.ones(vec_size, dtype=np.float)


10000
Hell naw I ain't done yet I L-B-see y'all prayin
15000
team is off the hook and you don't really want


In [186]:
print(X.shape, y.shape)

(17725, 10, 2903)

In [141]:
y.shape

(17725, 2903)

In [46]:
model=Sequential()
model.add(LSTM(128,input_shape=(num_words,vocab_size)))
model.add(Dense(vocab_size))
model.add(Activation('softmax'))
model.summary()
model.compile(optimizer=Adam(lr=0.0001),loss='categorical_crossentropy')

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 128)               154112    
                                                                 
 dense (Dense)               (None, 172)               22188     
                                                                 
 activation (Activation)     (None, 172)               0         
                                                                 
Total params: 176,300
Trainable params: 176,300
Non-trainable params: 0
_________________________________________________________________


c:\Python38\lib\site-packages\keras\optimizer_v2\adam.py:105: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(Adam, self).__init__(name, **kwargs)


In [47]:
model.fit(X,y,epochs=3,batch_size=256)

Epoch 1/3
116/116 [==============================] - 37s 287ms/step - loss: 4.4253
Epoch 2/3
116/116 [==============================] - 31s 269ms/step - loss: 3.1117
Epoch 3/3
116/116 [==============================] - 29s 248ms/step - loss: 3.0360


In [48]:
#serialize model to JSON  serialize model to JSON
# model_json = model.to_json()
# with open("model.json", "w") as json_file:
#     json_file.write(model_json)
# serialize weights to HDF5
model.save_weights("./../dataset/models/model_char.h5")
print("Saved model to disk")

Saved model to disk


In [52]:
# import random
np.random.seed(0)
# generated=''
# start_index=random.randint(0,len(txt)-maxlen-1)
# sent=txt[start_index:start_index+maxlen]
generated = "(Ugh) Ha-ha-ha, I'm serious, nigga One of".lower()
# actual_lyric = split_str(" ;,-'\\\'\"()[].?!:{}",  train_rap.loc[0]["lyrics"].lower().strip())
# actual_lyric = " ".join([s for s in actual_lyric if s!=''][0:50])
actual_lyric = train_rap.loc[0]["lyrics"][0:100]

print(actual_lyric)
# generated+=sent
for i in range(50):
    x_sample = generated[i:i+num_words]

    x=np.zeros((1,num_words,vocab_size))
    for w in range(num_words):
        x[0, w, char_ix[x_sample[w]]] = 1
    pred = model.predict(x)
    pred = np.reshape(pred, pred.shape[1])
    # ix = np.argmax(pred)
    # print(ix_char[np.argmax(pred)])

    ix=np.random.choice(range(vocab_size),p=pred.ravel())
    # print(ix, ix_char[ix])
    generated+= ix_char[ix]
print(generated)

(Ugh) Ha-ha-ha, I'm serious, nigga One of y'all niggas got some bad motherfuckin' breath (Oh, man) A
(ugh) ha-ha-ha, i'm serious, nigga one ofiolienetyeriiu   trtósgr m wieargi lllwneeo nn' ce


In [157]:
# embedd_index

In [ ]:
# embedd_index

In [ ]:
print(generated)